In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm
import numpy as np

# ======================
# Model Architecture
# ======================
class FERResNet18(nn.Module):
    def __init__(self, num_classes):
        super(FERResNet18, self).__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# ======================
# Device
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# Data Augmentation
# ======================
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(48, scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ======================
# Datasets
# ======================
train_dataset = datasets.ImageFolder(
    root="../datasets/FER_2013_enhanced/FER_preprocessed_train",
    transform=train_transform
)
test_dataset = datasets.ImageFolder(
    root="../datasets/FER_2013_enhanced/preprocessed_test",
    transform=test_transform
)

class_names = train_dataset.classes

# ======================
# Weighted Sampling
# ======================
targets = [label for _, label in train_dataset.samples]
class_count = np.bincount(targets)
class_weights = 1. / class_count
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# ======================
# DataLoaders
# ======================
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

# ======================
# Model, Loss, Optimizer, Scheduler
# ======================
model = FERResNet18(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# ======================
# Training & Evaluation
# ======================
def train_one_epoch(epoch):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}] Training", leave=False)

    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = 100. * correct / total
    print(f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%")
    return epoch_loss, epoch_acc

def evaluate():
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Evaluating", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    return acc

# ======================
# Main Training Loop
# ======================
num_epochs = 35
best_acc = 0

for epoch in range(num_epochs):
    train_one_epoch(epoch)
    val_acc = evaluate()
    scheduler.step()

    print(f"Validation Accuracy: {val_acc:.2f}%")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_fer_model.pth")
        print("🔥 Model Saved!")

print(f"Best Validation Accuracy: {best_acc:.2f}%")


Train Loss: 1.7047, Train Acc: 36.85%


Validation Accuracy: 45.22%
🔥 Model Saved!


Train Loss: 1.4850, Train Acc: 48.64%


Validation Accuracy: 49.35%
🔥 Model Saved!


Train Loss: 1.3804, Train Acc: 54.48%


...

Validation Accuracy: 61.86%


Train Loss: 0.8315, Train Acc: 82.10%


Validation Accuracy: 61.98%


Train Loss: 0.8316, Train Acc: 82.17%


Validation Accuracy: 61.95%
Best Validation Accuracy: 62.29%
